# JAX Gradient Examples for ZFEL

This notebook shows how to compute gradients of the **deterministic** FEL core using `sase_from_initial_conditions_jax(...)`.

Workflow:
1. Build `params` and fixed `bucket_data` once
2. Define a scalar loss from simulation outputs
3. Apply `jax.grad(loss_fn)`


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from zfel import sase1d

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


## 1) Build Fixed Initial Conditions

Use a modest grid so gradients evaluate quickly.


In [ ]:
sase_input = dict(
    npart=128,
    s_steps=32,
    z_steps=32,
    energy=4313.34e6,
    eSpread=0.01,
    emitN=1.2e-6,
    currentMax=3400,
    beta=26,
    unduPeriod=0.03,
    unduK=3.5,
    unduL=70,
    radWavelength=None,
    random_seed=31,
    particle_position=None,
    hist_rule='square-root',
    iopt='sase',
    P0=1e3,
)

params_np = sase1d.params_calc(**sase_input)
bucket_np = sase1d.fixed_or_external_bucket_data(
    params=params_np,
    npart=sase_input['npart'],
    s_steps=sase_input['s_steps'],
    particle_position=sase_input['particle_position'],
    hist_rule=sase_input['hist_rule'],
    iopt=sase_input['iopt'],
    random_seed=sase_input['random_seed'],
)

params = {k: jnp.asarray(v) for k, v in params_np.items()}
bucket_data = {k: (int(v) if k == 's_steps' else jnp.asarray(v)) for k, v in bucket_np.items()}

out0 = sase1d.sase_from_initial_conditions_jax(params, bucket_data)
print('Baseline final power_z [W]:', float(out0['power_z'][-1]))


## 2) Scalar Gradient: Seed Power (`E02`)

Differentiate final output power with respect to the scaled seed-power parameter `E02`.


In [ ]:
def loss_from_E02(E02):
    p = dict(params)
    p['E02'] = E02
    out = sase1d.sase_from_initial_conditions_jax(p, bucket_data)
    return out['power_z'][-1]

E02_test = params['E02'] + 1e-12
grad_E02 = jax.grad(loss_from_E02)(E02_test)

print('E02 test point:', float(E02_test))
print('d(final power_z)/dE02:', float(grad_E02))


## 3) Scalar Gradient: Scale Initial Energy Deviation

Scale all initial `eta_init` values by a scalar `alpha` and differentiate total emitted power.


In [ ]:
eta0 = bucket_data['eta_init']

def loss_from_eta_scale(alpha):
    b = dict(bucket_data)
    b['eta_init'] = alpha * eta0
    out = sase1d.sase_from_initial_conditions_jax(params, b)
    return jnp.sum(out['power_z'])

alpha0 = jnp.array(1.0)
grad_alpha = jax.grad(loss_from_eta_scale)(alpha0)
print('d(sum power_z)/dalpha at alpha=1:', float(grad_alpha))


In [ ]:
alphas = jnp.linspace(0.8, 1.2, 41)
loss_vals = jax.vmap(loss_from_eta_scale)(alphas)

plt.figure(figsize=(6,4))
plt.plot(np.asarray(alphas), np.asarray(loss_vals))
plt.axvline(1.0, color='k', ls='--', lw=1)
plt.xlabel('alpha (eta_init scale)')
plt.ylabel('sum(power_z) [W]')
plt.title('Loss vs eta_init scaling')
plt.grid(alpha=0.3)


## 4) Vector Gradient: Undulator Coupling Profile (`kappa_1`)

Treat `kappa_1` as an optimizable profile and compute `d(final power)/d(kappa_1[j])`.


In [ ]:
def loss_from_kappa1(kappa_1_profile):
    p = dict(params)
    p['kappa_1'] = kappa_1_profile
    out = sase1d.sase_from_initial_conditions_jax(p, bucket_data)
    return out['power_z'][-1]

kappa_1_0 = params['kappa_1']
grad_kappa1 = jax.grad(loss_from_kappa1)(kappa_1_0)

print('grad_kappa1 shape:', grad_kappa1.shape)
print('grad_kappa1 L2 norm:', float(jnp.linalg.norm(grad_kappa1)))


In [ ]:
z_idx = np.arange(len(np.asarray(grad_kappa1)))

fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(z_idx, np.asarray(kappa_1_0))
ax[0].set_title('kappa_1 profile')
ax[0].set_xlabel('z-step index')
ax[0].set_ylabel('kappa_1')
ax[0].grid(alpha=0.3)

ax[1].plot(z_idx, np.asarray(grad_kappa1))
ax[1].set_title('Gradient d(final power_z)/d(kappa_1[j])')
ax[1].set_xlabel('z-step index')
ax[1].set_ylabel('gradient')
ax[1].grid(alpha=0.3)

fig.tight_layout()


## 5) Gradient w.r.t. `unduK`

`unduK` influences multiple derived parameters (`kappa_1`, `Kai`, `deta`, resonance quantities).
This cell rebuilds those terms with JAX and differentiates final output power w.r.t. a scalar `unduK`.


In [ ]:
import math

# JAX in this environment does not expose scipy.special.jv directly.
# Use truncated series for J0/J1 so the full pipeline remains differentiable.

def j0_series(x, n_terms=20):
    x = jnp.asarray(x)
    acc = jnp.zeros_like(x)
    for m in range(n_terms):
        coeff = ((-1.0) ** m) / (math.factorial(m) ** 2)
        acc = acc + coeff * (x**2 / 4.0) ** m
    return acc


def j1_series(x, n_terms=20):
    x = jnp.asarray(x)
    acc = jnp.zeros_like(x)
    for m in range(n_terms):
        coeff = ((-1.0) ** m) / (math.factorial(m) * math.factorial(m + 1))
        acc = acc + coeff * (x / 2.0) ** (2 * m + 1)
    return acc


# Pull constants from module-level definitions used by params_calc
alfvenCurrent = sase1d.alfvenCurrent
mc2 = sase1d.mc2
c = sase1d.c
e = sase1d.e
epsilon_0 = sase1d.epsilon_0

# Base scalar settings
energy = jnp.asarray(sase_input['energy'])
emitN = jnp.asarray(sase_input['emitN'])
currentMax = jnp.asarray(sase_input['currentMax'])
beta = jnp.asarray(sase_input['beta'])
unduPeriod = jnp.asarray(sase_input['unduPeriod'])
radWavelength = params['resWavelength']
eSpread = jnp.asarray(sase_input['eSpread'])
P0 = jnp.asarray(sase_input['P0'])
z_steps = int(sase_input['z_steps'])


def params_from_unduK_scalar(unduK_scalar):
    unduK_profile = jnp.full((z_steps,), unduK_scalar)
    x = unduK_profile**2 / (4 + 2 * unduK_profile**2)
    unduJJ = j0_series(x) - j1_series(x)

    gamma0 = energy / mc2
    sigmaX2 = emitN * beta / gamma0

    kappa_1 = e * unduK_profile * unduJJ / 4 / epsilon_0 / gamma0
    Kai = e * unduK_profile * unduJJ / (2 * gamma0**2 * mc2 * e)
    density = currentMax / (e * c * 2 * jnp.pi * sigmaX2)

    resWavelength = unduPeriod * (1 + unduK_profile[0] ** 2 / 2.0) / (2 * gamma0**2)
    Pbeam = energy * currentMax
    coopLength = resWavelength / unduPeriod

    z0 = jnp.asarray(sase_input['unduL'])
    delt = z0 / z_steps
    dels = delt

    E02 = density * kappa_1[0] * P0 / Pbeam / Kai[0]
    gbar = resWavelength / radWavelength - 1.0
    Ns = currentMax * z0 / unduPeriod / z_steps * resWavelength / c / e
    deta = jnp.sqrt((1 + 0.5 * unduK_profile[0] ** 2) / (1 + 0.5 * unduK_profile**2)) - 1

    p = dict(params)
    p.update({
        'kappa_1': kappa_1,
        'Kai': Kai,
        'density': density,
        'resWavelength': resWavelength,
        'coopLength': coopLength,
        'delt': delt,
        'dels': dels,
        'E02': E02,
        'gbar': gbar,
        'Ns': Ns,
        'deta': deta,
    })
    return p


def loss_from_unduK(unduK_scalar):
    p = params_from_unduK_scalar(unduK_scalar)
    out = sase1d.sase_from_initial_conditions_jax(p, bucket_data)
    return out['power_z'][-1]


unduK0 = jnp.asarray(sase_input['unduK'])
grad_unduK = jax.grad(loss_from_unduK)(unduK0)

print('unduK baseline:', float(unduK0))
print('d(final power_z)/d(unduK):', float(grad_unduK))


## 6) Gradient w.r.t. Array `unduK` (Taper Profile)

This mirrors the taper setting where `unduK` is a full profile `K[z]`.
We differentiate final power with respect to every element in `K_profile`.


In [ ]:
# A simple linear taper profile example
K_profile0 = jnp.linspace(3.6, 3.2, z_steps)

def params_from_unduK_profile(K_profile):
    x = K_profile**2 / (4 + 2 * K_profile**2)
    unduJJ = j0_series(x) - j1_series(x)

    gamma0 = energy / mc2
    sigmaX2 = emitN * beta / gamma0

    kappa_1 = e * K_profile * unduJJ / 4 / epsilon_0 / gamma0
    Kai = e * K_profile * unduJJ / (2 * gamma0**2 * mc2 * e)
    density = currentMax / (e * c * 2 * jnp.pi * sigmaX2)

    resWavelength = unduPeriod * (1 + K_profile[0] ** 2 / 2.0) / (2 * gamma0**2)
    Pbeam = energy * currentMax
    coopLength = resWavelength / unduPeriod

    z0 = jnp.asarray(sase_input['unduL'])
    delt = z0 / z_steps
    dels = delt

    E02 = density * kappa_1[0] * P0 / Pbeam / Kai[0]
    gbar = resWavelength / radWavelength - 1.0
    Ns = currentMax * z0 / unduPeriod / z_steps * resWavelength / c / e
    deta = jnp.sqrt((1 + 0.5 * K_profile[0] ** 2) / (1 + 0.5 * K_profile**2)) - 1

    p = dict(params)
    p.update({
        'kappa_1': kappa_1,
        'Kai': Kai,
        'density': density,
        'resWavelength': resWavelength,
        'coopLength': coopLength,
        'delt': delt,
        'dels': dels,
        'E02': E02,
        'gbar': gbar,
        'Ns': Ns,
        'deta': deta,
    })
    return p

def loss_from_unduK_profile(K_profile):
    p = params_from_unduK_profile(K_profile)
    out = sase1d.sase_from_initial_conditions_jax(p, bucket_data)
    return out['power_z'][-1]

grad_K_profile = jax.grad(loss_from_unduK_profile)(K_profile0)

print('K_profile shape:', K_profile0.shape)
print('grad shape:', grad_K_profile.shape)
print('grad L2 norm:', float(jnp.linalg.norm(grad_K_profile)))


In [ ]:
z_idx = np.arange(z_steps)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(z_idx, np.asarray(K_profile0))
ax[0].set_title('Taper profile K[z]')
ax[0].set_xlabel('z-step index')
ax[0].set_ylabel('K')
ax[0].grid(alpha=0.3)

ax[1].plot(z_idx, np.asarray(grad_K_profile))
ax[1].set_title('Gradient d(final power_z)/dK[z]')
ax[1].set_xlabel('z-step index')
ax[1].set_ylabel('gradient')
ax[1].grid(alpha=0.3)

fig.tight_layout()


## Notes

- Gradients are only for the deterministic solver path (`sase_from_initial_conditions_jax`).
- Keep `bucket_data` fixed while differentiating model parameters.
- For optimization, wrap your scalar objective and feed gradients to your optimizer.
